# Bispectrum models, terms, and representations

A fastnc bispectrum is a sum of named physical terms. A term can carry several mathematically equivalent representations, allowing downstream calculators to select an algorithm without changing the model. Representations attached to one term are alternatives, not additive pieces.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from fastnc.bispectrum import (
    Bispectrum2D, BispectrumTerm2D, NumericExpression2D,
    SPTMatterBispectrum3D, TriangleInterpolationConfig,
)

## Inspect and evaluate a built-in 3D model

In [ ]:
b3d = SPTMatterBispectrum3D.simple_debug()
for term in b3d.terms:
    print(term.name, [type(rep).__name__ for rep in term.representations])

k = np.geomspace(0.02, 0.5, 64)
equilateral = b3d.evaluate(k, k, k, z=0.5)
plt.loglog(k, np.abs(equilateral))
plt.xlabel(r"$k$")
plt.ylabel(r"$|B(k,k,k;z=0.5)|$");

Numeric evaluation is the common reference interface. A coefficient may be callable as coefficient(z), allowing redshift-dependent amplitudes without embedding them in every representation.

## Select, scale, and combine terms

In [ ]:
name = b3d.terms[0].name
first = b3d.select_terms(name)
combined = first + 0.5 * first
reference = first.evaluate(0.1, 0.1, 0.1, z=0.5)
assert np.allclose(combined.evaluate(0.1, 0.1, 0.1, z=0.5), 1.5 * reference)

Selection preserves source-state tracking. Addition creates a numeric composite, which is useful when several terms should be interpolated together.

## Define a native angular bispectrum

In [ ]:
def toy_expression(ell1, ell2, ell3):
    ell1, ell2, ell3 = np.broadcast_arrays(ell1, ell2, ell3)
    return np.exp(-(ell1 + ell2 + ell3) / 2000.0)

toy_term = BispectrumTerm2D(
    name="tutorial:toy",
    representations=(NumericExpression2D(toy_expression),),
)
toy_b2d = Bispectrum2D((toy_term,))
print(toy_b2d.evaluate(100.0, 120.0, 80.0))

Native and LOS-projected angular models expose the same evaluate(ell1, ell2, ell3) interface.

## Interpolate the numeric representation

In [ ]:
interpolation = TriangleInterpolationConfig(
    ell2=np.geomspace(20.0, 2000.0, 32),
    ell3=np.geomspace(20.0, 2000.0, 32),
    mu23=np.linspace(-0.999, 1.0, 33),
)
interpolated = toy_b2d.interpolate(interpolation, prepare=True)
direct = toy_b2d.evaluate(100.0, 120.0, 80.0)
cached = interpolated.evaluate(100.0, 120.0, 80.0)
print(direct, cached, abs(cached / direct - 1.0))

Interpolation is a numeric representation with an owned cache and source-state token. Structured representations remain attached to the term.